In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/survey lung cancer.csv")
print("Dimension of dataset: \n")
print(df.shape)

Dimension of dataset: 

(309, 16)


In [ ]:
df.head()

,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,M,69,1,2,2,1,1,2,1,2,2,2,2,2,2,YES
1,M,74,2,1,1,1,2,2,2,1,1,1,2,2,2,YES
2,F,59,1,1,1,2,1,2,1,2,1,2,2,1,2,NO
3,M,63,2,2,2,1,1,1,1,1,2,1,1,2,2,NO
4,F,63,1,2,1,1,1,1,1,2,1,2,2,1,1,NO


In [ ]:
print(df.columns)

Index(['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY',
       'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE ', 'ALLERGY ', 'WHEEZING',
       'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH',
       'SWALLOWING DIFFICULTY', 'CHEST PAIN', 'LUNG_CANCER'],
      dtype='object')


In [ ]:
df['GENDER'] = df['GENDER'].map({'M':1, 'F':0})
df['LUNG_CANCER'] = df['LUNG_CANCER'].map({'YES':1, 'NO':0})

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   GENDER                 309 non-null    int64
 1   AGE                    309 non-null    int64
 2   SMOKING                309 non-null    int64
 3   YELLOW_FINGERS         309 non-null    int64
 4   ANXIETY                309 non-null    int64
 5   PEER_PRESSURE          309 non-null    int64
 6   CHRONIC DISEASE        309 non-null    int64
 7   FATIGUE                309 non-null    int64
 8   ALLERGY                309 non-null    int64
 9   WHEEZING               309 non-null    int64
 10  ALCOHOL CONSUMING      309 non-null    int64
 11  COUGHING               309 non-null    int64
 12  SHORTNESS OF BREATH    309 non-null    int64
 13  SWALLOWING DIFFICULTY  309 non-null    int64
 14  CHEST PAIN             309 non-null    int64
 15  LUNG_CANCER            309 non-null    i

In [ ]:
len(df["AGE"].unique())

39

In [ ]:
col = list(df.columns)
print(col)

['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE ', 'ALLERGY ', 'WHEEZING', 'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH', 'SWALLOWING DIFFICULTY', 'CHEST PAIN', 'LUNG_CANCER']


In [ ]:
col.remove("LUNG_CANCER")
col.remove("GENDER")
col.remove("AGE")
print(col)

['SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE ', 'ALLERGY ', 'WHEEZING', 'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH', 'SWALLOWING DIFFICULTY', 'CHEST PAIN']


In [ ]:
for c in col:
  df[c] = df[c].map({1:0, 2:1})

In [ ]:
df.isna().sum()

,0
GENDER,0
AGE,0
SMOKING,0
YELLOW_FINGERS,0
ANXIETY,0
PEER_PRESSURE,0
CHRONIC DISEASE,0
FATIGUE,0
ALLERGY,0
WHEEZING,0


In [ ]:
X = df.drop("LUNG_CANCER", axis=1)
y = df["LUNG_CANCER"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
X_train.shape

(216, 15)

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled[X_train_scaled < 0] = 0
X_test_scaled[X_test_scaled < 0] = 0

In [ ]:
nmf = NMF(n_components=4, init='nndsvda', random_state=42)

X_train_nmf = nmf.fit_transform(X_train_scaled)
X_test_nmf = nmf.transform(X_test_scaled)

In [ ]:
X_train_nmf.shape

(216, 4)

In [62]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [63]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')

clf.fit(X_train_nmf, y_train)
y_pred = clf.predict(X_test_nmf)

In [64]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9247311827956989

Confusion Matrix:
 [[ 9  3]
 [ 4 77]]

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.75      0.72        12
           1       0.96      0.95      0.96        81

    accuracy                           0.92        93
   macro avg       0.83      0.85      0.84        93
weighted avg       0.93      0.92      0.93        93



In [65]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=4)
dt_clf.fit(X_train_nmf, y_train)

y_pred = dt_clf.predict(X_test_nmf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8064516129032258

Confusion Matrix:
 [[10  2]
 [16 65]]

Classification Report:
               precision    recall  f1-score   support

           0       0.38      0.83      0.53        12
           1       0.97      0.80      0.88        81

    accuracy                           0.81        93
   macro avg       0.68      0.82      0.70        93
weighted avg       0.89      0.81      0.83        93



In [66]:
import xgboost as xgb

pos_count = sum(y_train == 1)
neg_count = sum(y_train == 0)
scale_pos_weight = neg_count / pos_count

In [69]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_clf.fit(X_train_nmf, y_train)
y_pred = xgb_clf.predict(X_test_nmf)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [18:04:19] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [70]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9247311827956989

Confusion Matrix:
 [[11  1]
 [ 6 75]]

Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.92      0.76        12
           1       0.99      0.93      0.96        81

    accuracy                           0.92        93
   macro avg       0.82      0.92      0.86        93
weighted avg       0.94      0.92      0.93        93

